In [1]:
import pandas as pd
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


In [2]:
df = pd.read_csv('sales_data.csv', parse_dates=['Date'], dayfirst=True)



In [3]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True,format='ISO8601')


In [4]:
df_filled = df.fillna(0)

In [5]:
print(df.isnull().sum())

Date               0
Product_ID         0
Product_Name       0
Category           0
Quantity_Sold      0
Wholesale_Price    0
Retail_Price       0
Revenue            0
Profit             0
dtype: int64


In [6]:
print(df.dtypes)

Date               datetime64[ns]
Product_ID                  int64
Product_Name               object
Category                   object
Quantity_Sold               int64
Wholesale_Price             int64
Retail_Price                int64
Revenue                     int64
Profit                      int64
dtype: object


In [7]:
# Display summary statistics
print(df.describe())


                      Date   Product_ID  Quantity_Sold  Wholesale_Price  \
count                 3960  3960.000000    3960.000000      3960.000000   
mean   2023-07-17 12:00:00     5.500000      19.389141       399.000000   
min    2023-01-01 00:00:00     1.000000      10.000000       110.000000   
25%    2023-04-09 18:00:00     3.000000      14.000000       300.000000   
50%    2023-07-17 12:00:00     5.500000      19.000000       400.000000   
75%    2023-10-24 06:00:00     8.000000      24.000000       560.000000   
max    2024-01-31 00:00:00    10.000000      29.000000       700.000000   
std                    NaN     2.872644       5.819186       177.755391   

       Retail_Price       Revenue       Profit  
count   3960.000000   3960.000000  3960.000000  
mean     473.000000   9181.739646  1437.949242  
min      159.000000   1590.000000   390.000000  
25%      399.000000   5388.000000   931.000000  
50%      469.000000   8778.000000  1274.000000  
75%      669.000000  11998.500

In [8]:
df = df.dropna()

In [9]:
df_filled['Day_of_Week'] = df_filled['Date'].dt.dayofweek
df_filled['Month_of_Year'] = df_filled['Date'].dt.month


In [10]:
# Check if 'Product_ID' is present in your dataset
if 'Product_ID' in df_filled.columns:
    print("Product_ID is present in the dataset.")
else:
    print("Product_ID is missing from the dataset.")


Product_ID is present in the dataset.


In [11]:
# Apply one-hot encoding to the 'Product_Name' and 'Product_ID' columns
preprocessor = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), ['Product_Name', 'Product_ID']),
        ('numeric', 'passthrough', ['Day_of_Week', 'Month_of_Year'])
    ],
    remainder='drop'
)


In [12]:
X = preprocessor.fit_transform(df_filled.drop(['Quantity_Sold', 'Date'], axis=1))
y = df_filled['Quantity_Sold']

In [13]:

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)


RandomForestRegressor(random_state=42)

In [14]:
current_date = datetime.now().date()
print(current_date)

2024-05-02


In [15]:
product_ids = df_filled['Product_ID'].unique()
for product_id in product_ids:
    # Extract product name
    product_name = df_filled.loc[df_filled['Product_ID'] == product_id, 'Product_Name'].iloc[0]

    # Adjust X_future with relevant feature values for the future period
    X_future = preprocessor.transform(pd.DataFrame({
        'Product_Name': [product_name],  # Specify the product for forecasting
        'Product_ID': [product_id],  # Use the existing Product_ID
        'Day_of_Week': [current_date.weekday()],  # Use the current day of the week
        'Month_of_Year': [current_date.month],  # Use the current month
    }))

    # Make predictions for the next one month
    future_date = current_date + timedelta(days=30)
    future_date_features = pd.DataFrame({
        'Product_Name': [product_name],  # Specify the product for forecasting
        'Product_ID': [product_id],  # Use the existing Product_ID
        'Day_of_Week': [future_date.weekday()],  # Adjust with future date
        'Month_of_Year': [future_date.month],  # Adjust with future date
    })
    X_future = preprocessor.transform(future_date_features)

    # Make predictions for the next one month
    forecasted_quantity = rf_model.predict(X_future)

    # Round the forecasted quantity to the nearest whole number
    rounded_forecast = round(forecasted_quantity[0])

    # Print or use the rounded_forecast as needed
    print(f"Estimated sales of the ({product_id}){product_name} for the next one month: {rounded_forecast}")


Estimated sales of the (1)Polo shirt for the next one month: 16
Estimated sales of the (2)Jeans for the next one month: 20
Estimated sales of the (3)Denim jacket for the next one month: 21
Estimated sales of the (4)Tuxedo for the next one month: 20
Estimated sales of the (5)V-neck -Sweaters for the next one month: 19
Estimated sales of the (6)Graphic T-shirt for the next one month: 26
Estimated sales of the (7)Running jacket for the next one month: 21
Estimated sales of the (8)Belt for the next one month: 21
Estimated sales of the (9)Watch for the next one month: 22
Estimated sales of the (10)Sneakers for the next one month: 20
